# 02 Load Daily City Metrics

* Author: Jeremiah Hansen
* Last Updated: 2/12/2026

This notebook will load data into the `DAILY_CITY_METRICS` table with support for incremental processing.

In [ ]:
# Import python packages
import sys
import logging

# Set up the logger
logger_name = 'demo_logger'
logger = logging.getLogger(logger_name)
logger.setLevel(logging.INFO)

# Set default values for debugging
notebook_name = '02_load_daily_city_metrics.ipynb'
database_name = 'DEMO_DB'
schema_name = 'DEV_SCHEMA'
role_name = 'DEMO_ROLE'

# Override values with passed notebook arguments
if sys.argv[0].endswith('.ipynb'):
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument('--database-name', type=str)
    parser.add_argument('--schema-name', type=str)
    parser.add_argument('--role-name', type=str)
    args, args_unknown = parser.parse_known_args()

    notebook_name = parser.prog  # same as argv[0]
    database_name = args.database_name or database_name
    schema_name = args.schema_name or schema_name
    role_name = args.role_name or role_name

# Get a Snowpark session
from snowflake.snowpark.context import get_active_session
session = get_active_session()

# Set the default database and schema for the following cells
session.use_schema(f"{database_name}.{schema_name}")

# Set the role
# Needed when running EXECUTE NOTEBOOK PROJECT directly (since it ignores session context and uses the user's default role)
session.use_role(role_name)

# Get details about the current state
current_state_df = session.sql(f"""
        SELECT OBJECT_CONSTRUCT(
            'current_user', CURRENT_USER(),
            'current_role', CURRENT_ROLE(),
            'current_secondary_roles', PARSE_JSON(CURRENT_SECONDARY_ROLES()),
            'current_database', CURRENT_DATABASE(),
            'current_schema', CURRENT_SCHEMA()
        )::STRING AS session_context;
    """).collect()

logger.info(f"Begin executing notebook {notebook_name}", extra = {'logger_name': logger_name})
logger.info(f"Using parameters database: {database_name}, schema: {schema_name}, role: {role_name}", extra = {'logger_name': logger_name})
logger.info(f"Using session context {current_state_df[0]['SESSION_CONTEXT']}", extra = {'logger_name': logger_name})

## Create a function to check if a table exists

This function uses the [Snowflake Python Management API](https://docs.snowflake.com/en/developer-guide/snowflake-python-api/snowflake-python-overview).

In [ ]:
def table_exists(session, database_name='', schema_name='', table_name=''):
    from snowflake.core import Root

    root = Root(session)
    tables = root.databases[database_name].schemas[schema_name].tables.iter(like=table_name)
    for table_obj in tables:
        if table_obj.name == table_name:
            return True

    return False

# Not used, SQL alternative to Python version above
def table_exists2(session, database_name='', schema_name='', table_name=''):
    exists = session.sql("SELECT EXISTS (SELECT * FROM {}.INFORMATION_SCHEMA.TABLES WHERE TABLE_SCHEMA = '{}' AND TABLE_NAME = '{}') AS TABLE_EXISTS".format(database_name, schema_name, table_name)).collect()[0]['TABLE_EXISTS']
    return exists

## Pipeline to update daily_city_metrics

In [ ]:
import snowflake.snowpark.functions as F

table_name = "DAILY_CITY_METRICS"

# Define the tables
order_detail = session.table("ORDER_DETAIL")
history_day = session.table("FROSTBYTE_WEATHERSOURCE.ONPOINT_ID.HISTORY_DAY")
location = session.table("LOCATION")

# Join the tables
order_detail = order_detail.join(
    location,
    order_detail['LOCATION_ID'] == location['LOCATION_ID']
)

order_detail = order_detail.join(
    history_day,
    (F.builtin("DATE")(order_detail['ORDER_TS']) == history_day['DATE_VALID_STD'])
    & (location['ISO_COUNTRY_CODE'] == history_day['COUNTRY'])
)

# Aggregate the data
final_agg = (
    order_detail.group_by(
        F.col('DATE_VALID_STD'),
        F.col('CITY'),
        F.col('ISO_COUNTRY_CODE')
    )
    .agg(
        F.sum('PRICE').alias('DAILY_SALES_SUM'),
        F.avg('AVG_TEMPERATURE_AIR_2M_C').alias('AVG_TEMPERATURE_C'),
        F.avg('TOT_PRECIPITATION_MM').alias('AVG_PRECIPITATION_MM'),
    )
    .select(
        F.col("DATE_VALID_STD").alias("DATE"),
        F.col("CITY"),
        F.col("ISO_COUNTRY_CODE").alias("COUNTRY_DESC"),
        F.builtin("ZEROIFNULL")(F.col("DAILY_SALES_SUM")).alias("DAILY_SALES"),
        F.round(F.col("AVG_TEMPERATURE_C"), 2).alias("AVG_TEMPERATURE_C"),
        F.round(F.col("AVG_PRECIPITATION_MM"), 2).alias("AVG_PRECIPITATION_MM"),
    )
)

# If the table doesn't exist then create it
if not table_exists(
    session,
    database_name=database_name,
    schema_name=schema_name,
    table_name=table_name
):
    final_agg.write.mode("overwrite").save_as_table(table_name)

    logger.info(
        f"Successfully created {table_name}",
        extra={'logger_name': logger_name}
    )

# Otherwise update it
else:
    cols_to_update = {c: final_agg[c] for c in final_agg.schema.names}

    dcm = session.table(table_name)

    dcm.merge(
        final_agg,
        (dcm['DATE'] == final_agg['DATE'])
        & (dcm['CITY'] == final_agg['CITY'])
        & (dcm['COUNTRY_DESC'] == final_agg['COUNTRY_DESC']),
        [
            F.when_matched().update(cols_to_update),
            F.when_not_matched().insert(cols_to_update)
        ]
    )

    logger.info(
        f"Successfully updated {table_name}",
        extra={'logger_name': logger_name}
    )

logger.info(
    f"Finish executing notebook {notebook_name}",
    extra={'logger_name': logger_name}
)

## Debugging

In [ ]:
%%sql -r dataframe_1
--SELECT * FROM DAILY_CITY_METRICS LIMIT 10;

In [ ]:
import snowflake.snowpark.functions as F

# Top 10 cities by total sales from DAILY_CITY_METRICS
top_cities = (
    session.table("DAILY_CITY_METRICS")
    .group_by("CITY", "COUNTRY_DESC")
    .agg(
        F.sum("DAILY_SALES").alias("TOTAL_SALES"),
        F.count_distinct("DATE").alias("TOTAL_DAYS"),
        F.round(F.avg("AVG_TEMPERATURE_C"), 2).alias("AVG_TEMPERATURE_C"),
    )
    .sort(F.col("TOTAL_SALES").desc())
    .limit(10)
)

top_cities.show()

In [ ]:
history_day.columns

In [ ]:
location.columns

In [ ]:
session.table("FROSTBYTE_WEATHERSOURCE.ONPOINT_ID.HISTORY_DAY").limit(5).show()

In [ ]:
session.sql("""
SHOW VIEWS IN SCHEMA FROSTBYTE_WEATHERSOURCE.ONPOINT_ID
""").show()

In [ ]:
%%sql -r dataframe_2
SELECT 
    TIMESTAMP,
    VALUE AS LOG_MESSAGE,
    RESOURCE_ATTRIBUTES:"snow.service.name"::string AS SERVICE_NAME,
    RECORD_ATTRIBUTES:"severity_text"::string AS SEVERITY
FROM DEMO_DB.INTEGRATIONS.DEMO_EVENTS
WHERE RECORD_TYPE = 'LOG'
--  AND RESOURCE_ATTRIBUTES:"snow.service.name" = 'NOTEBOOK_SERVICE'
  AND TIMESTAMP > DATEADD(hour, -1, CURRENT_TIMESTAMP())
ORDER BY TIMESTAMP DESC
LIMIT 100;
